In [ ]:
import logging

FORMAT = '%(asctime)s %(name)s %(funcName)s %(message)s'
log_level = logging.WARNING
logging.basicConfig(format=FORMAT, datefmt='%H:%M:%S',
                    level=log_level)

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
import h5py
import numpy as np
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import csv
from pathlib import Path 

bnsi_path = '/scicore/home/nimwegen/degroo0000/Bonsai-data-representation/'
sys.path.append(bnsi_path)
from bonsai_scout.bonsai_scout_helpers import Bonvis_figure, Bonvis_settings, Bonvis_metadata

#### Set some parameters

In [ ]:
SINGLE_DATASET = False
N_PCS_UMAP = -1
N_PCS_PHATE = 100
N_PCS_DTNE = 500

methods = ['bonsai', 'pca', 'umap', 'phate', 'DTNE',  'sanity', 'bonsai-log1p']

SAVE_FIG = False

In [ ]:
PATH_TO_PAPER_FIGURES = "/scicore/home/nimwegen/GROUP/Projects/bonsai_runs"

#### List information on datasets

In [ ]:
if not SINGLE_DATASET:
    dataset_ids = ['simulated_datasets/simulated_binary_10_gens_samplingnoise_seed_2462',
                   'simulated_datasets/simulated_pseudobulk_based_ncells_1024_seed_1231',
    'simulated_datasets/simulated_binary_10_gens_samplingnoise_unbalanced_seed_2462',
    'simulated_datasets/simulated_binary_10_gens_samplingnoise_randomtimes_seed_2462',
    'simulated_datasets/simulated_binary_10_gens_samplingnoise_realcovariance_seed_2462',
    'simulated_datasets/simulated_binary_10_gens_samplingnoise_randomtimes_unbalanced_seed_2462',
    'simulated_datasets/simulated_binary_10_gens_samplingnoise_randomtimes_unbalanced_realcovariance_seed_2462']
else:
    dataset_ids = ['simulated_datasets/simulated_binary_10_gens_samplingnoise_randomtimes_unbalanced_realcovariance_seed_2462']

    
input_data_folders = [os.path.join(PATH_TO_PAPER_FIGURES, 'paper_figures_datasets', dataset_id, 'UMI_counts') for dataset_id in dataset_ids]
bonsai_results_folders = [os.path.join(PATH_TO_PAPER_FIGURES, 'paper_figures_datasets', dataset_id, 'bonsai') 
                          for dataset_id in dataset_ids]


bonsai_logp1_results_folders = [os.path.join(PATH_TO_PAPER_FIGURES, 'paper_figures_datasets', dataset_id, 'processed_raw_cnts_log1p_hvg/bonsai') 
                          for dataset_id in dataset_ids]


if not SINGLE_DATASET:
    dataset_descr_lst = ['Perfect binary', 'Pseudobulk', 'Unbalanced (Unb)', 'Random branch lengths (Rbl)', 'Real covariance (Reco)', 'Rbl + Unb', 'Rbl + Unb + Reco']
else:
    dataset_descr_lst = ['Rbl + Unb + Reco']


In [ ]:
print(bonsai_logp1_results_folders[0])
os.path.exists(bonsai_logp1_results_folders[0])

## Create Bonsai visualization of dataset

In [ ]:
os.path.join(bonsai_results, 'bonsai_vis_data.hdf')

In [ ]:
%%capture  
# The above %%capture is used for not showing the tree-visualizations that are created.
bonvis_metadata_lst = []
bonvis_settings_lst = []
bonvis_data_hdf_lst = []
bonvis_fig_lst = []
for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
    # Load metadata, settings and data
    print(bonsai_results)
    data_path = os.path.join(bonsai_results, 'bonsai_vis_data.hdf')
    settings_path = os.path.join(bonsai_results, 'bonsai_vis_settings.json')
    bonvis_metadata = Bonvis_metadata(data_path)
    bonvis_settings = Bonvis_settings(load_settings_path=settings_path)
    bonvis_data_hdf = h5py.File(data_path, 'r')
    bonvis_fig = Bonvis_figure(bonvis_data_hdf, bonvis_metadata, bonvis_data_path=data_path,
                           bonvis_settings=bonvis_settings)
    bonvis_fig.create_figure(figsize=(6, 6))

    bonvis_metadata_lst.append(bonvis_metadata)
    bonvis_settings_lst.append(bonvis_settings)
    bonvis_data_hdf_lst.append(bonvis_data_hdf)
    bonvis_fig_lst.append(bonvis_fig)

In [ ]:
%%capture  
# It is possible to create circular layout
for ind_dataset, bonvis_fig in enumerate(bonvis_fig_lst):
    bonvis_fig.update_figure(ly_type='ly_eq_angle')
    bonvis_fig.create_figure(figsize=(6, 6))

In [ ]:
# Here, we set the desired celltype-annotation for every dataset
node_style_lst = []
for ind_dataset, dataset_descr in enumerate(dataset_descr_lst):
    node_style = 'Celltype3' if (dataset_descr != "Pseudobulk") else 'Pseudobulk'
    node_style_lst.append(node_style)

In [ ]:
# Visualize the tree in the equal-daylight layout, with the correct celltype-annotation
for ind_dataset, bonvis_fig in enumerate(bonvis_fig_lst):
    node_style = node_style_lst[ind_dataset]
    geometry = 'flat' if (dataset_descr_lst[ind_dataset] == 'Pseudobulk') else 'hyperbolic'
    zoom = 1 if (dataset_descr_lst[ind_dataset] == 'Pseudobulk') else 1
    bonvis_fig.update_figure(ly_type='ly_eq_angle', geometry=geometry, node_style=node_style, zoom=zoom);

### Create Bonsai visualization of dataset with standard preprocessing

In [ ]:
%%capture  
# The above %%capture is used for not showing the tree-visualizations that are created.
bonvis_logp1_metadata_lst = []
bonvis_logp1_settings_lst = []
bonvis_logp1_data_hdf_lst = []
bonvis_logp1_fig_lst = []
for ind_dataset, bonsai_results in enumerate(bonsai_logp1_results_folders):
    # Load metadata, settings and data
    data_path = os.path.join(bonsai_results, 'bonsai_vis_data.hdf')
    settings_path = os.path.join(bonsai_results, 'bonsai_vis_settings.json')
    bonvis_metadata = Bonvis_metadata(data_path)
    bonvis_settings = Bonvis_settings(load_settings_path=settings_path)
    bonvis_data_hdf = h5py.File(data_path, 'r')
    bonvis_fig = Bonvis_figure(bonvis_data_hdf, bonvis_metadata, bonvis_data_path=data_path,
                           bonvis_settings=bonvis_settings)
    bonvis_fig.create_figure(figsize=(6, 6))

    bonvis_logp1_metadata_lst.append(bonvis_metadata)
    bonvis_logp1_settings_lst.append(bonvis_settings)
    bonvis_logp1_data_hdf_lst.append(bonvis_data_hdf)
    bonvis_logp1_fig_lst.append(bonvis_fig)

In [ ]:
%%capture  
# It is possible to create circular layout
for ind_dataset, bonvis_fig in enumerate(bonvis_logp1_fig_lst):
    bonvis_fig.update_figure(ly_type='ly_eq_angle')
    bonvis_fig.create_figure(figsize=(6, 6))

In [ ]:
# Here, we set the desired celltype-annotation for every dataset
node_style_lst = []
for ind_dataset, dataset_descr in enumerate(dataset_descr_lst):
    node_style = 'Celltype3' if (dataset_descr != "Pseudobulk") else 'Pseudobulk'
    node_style_lst.append(node_style)

In [ ]:
# Visualize the tree in the equal-daylight layout, with the correct celltype-annotation
for ind_dataset, bonvis_fig in enumerate(bonvis_logp1_fig_lst):
    node_style = node_style_lst[ind_dataset]
    geometry = 'flat' if (dataset_descr_lst[ind_dataset] == 'Pseudobulk') else 'hyperbolic'
    zoom = 1 if (dataset_descr_lst[ind_dataset] == 'Pseudobulk') else 1
    bonvis_fig.update_figure(ly_type='ly_eq_angle', geometry=geometry, node_style=node_style, zoom=zoom);

## Create Bonsai visualization of ground truth dataset

In [ ]:
if not SINGLE_DATASET:
    dataset_ids_gt = ['simulated_datasets/simulated_binary_10_gens_samplingnoise_seed_2462',
    'simulated_datasets/simulated_binary_10_gens_samplingnoise_unbalanced_seed_2462',
    'simulated_datasets/simulated_binary_10_gens_samplingnoise_randomtimes_seed_2462',
    'simulated_datasets/simulated_binary_10_gens_samplingnoise_realcovariance_seed_2462',
    'simulated_datasets/simulated_binary_10_gens_samplingnoise_randomtimes_unbalanced_seed_2462',
    'simulated_datasets/simulated_binary_10_gens_samplingnoise_randomtimes_unbalanced_realcovariance_seed_2462']

    dataset_descr_lst_gt = ['Perfect binary', 'Unbalanced (Unb)', 'Random branch lengths (Rbl)', 'Real covariance (Reco)', 'Rbl + Unb', 'Rbl + Unb + Reco']
else:
    dataset_ids_gt = ['simulated_datasets/simulated_binary_10_gens_samplingnoise_randomtimes_unbalanced_realcovariance_seed_2462']
    # dataset_ids_gt = []
    
    dataset_descr_lst_gt = ['Rbl + Unb + Reco']
    # dataset_descr_lst_gt = []

input_data_folders_gt = [os.path.join(PATH_TO_PAPER_FIGURES, 'paper_figures_datasets', id, 'UMI_counts') for id in dataset_ids_gt]
bonsai_results_folders_gt = [os.path.join(PATH_TO_PAPER_FIGURES, 'paper_figures_datasets', id, 'UMI_counts', 'true_tree') for id in dataset_ids_gt]



In [ ]:
%%capture  
# The above %%capture is used for not showing the tree-visualizations that are created.
bonvis_metadata_lst_gt = []
bonvis_settings_lst_gt = []
bonvis_data_hdf_lst_gt = []
bonvis_fig_lst_gt = []
for ind_dataset, bonsai_results in enumerate(bonsai_results_folders_gt):
    # Load metadata, settings and data
    data_path = os.path.join(bonsai_results, 'bonsai_vis_data.hdf')
    print(data_path)
    settings_path = os.path.join(bonsai_results, 'bonsai_vis_settings.json')
    bonvis_metadata = Bonvis_metadata(data_path)
    bonvis_settings = Bonvis_settings(load_settings_path=settings_path)
    bonvis_data_hdf = h5py.File(data_path, 'r')
    bonvis_fig = Bonvis_figure(bonvis_data_hdf, bonvis_metadata, bonvis_data_path=data_path,
                           bonvis_settings=bonvis_settings)
    bonvis_fig.create_figure(figsize=(6, 6))

    bonvis_metadata_lst_gt.append(bonvis_metadata)
    bonvis_settings_lst_gt.append(bonvis_settings)
    bonvis_data_hdf_lst_gt.append(bonvis_data_hdf)
    bonvis_fig_lst_gt.append(bonvis_fig)

In [ ]:
# Here, we set the desired celltype-annotation for every dataset
node_style_lst_gt = []
for ind_dataset, dataset_descr in enumerate(dataset_descr_lst_gt):
    node_style = 'Celltype3' if (dataset_descr != "Pseudobulk") else 'Pseudobulk'
    node_style_lst_gt.append(node_style)

In [ ]:
# Visualize the tree in the equal-daylight layout, with the correct celltype-annotation
for ind_dataset_gt, bonvis_fig in enumerate(bonvis_fig_lst_gt):
    ind_dataset = dataset_descr_lst.index(dataset_descr_lst_gt[ind_dataset_gt])
    node_style = node_style_lst_gt[ind_dataset_gt]
    bonvis_fig.update_figure(ly_type='ly_eq_angle', geometry='hyperbolic', node_style=node_style);

## Create PCA and UMAP plot of all datasets

In [ ]:
from paper_figure_scripts_and_notebooks.simulating_datasets.analyzing_simulated_datasets.knn_recall_helpers import do_pca, do_logp1, fit_umap, fit_phate, \
    fit_DTNE, get_pdists_on_tree, Dataset, \
    compare_nearest_neighbours_to_truth, compare_pdists_to_truth, get_pdists_on_tree, compare_pdists_to_truth_per_cell, compare_nearest_neighbours_to_truth
from bonsai.bonsai_helpers import find_latest_tree_folder
from scipy.spatial import distance
from bonsai.bonsai_dataprocessing import SCData, get_bonsai_euclidean_distances
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from matplotlib import colormaps
plt.set_loglevel(level='warning')

In [ ]:
umi_counts_lst = []
cell_ids_lst = []
for ind_dataset, input_data_folder in enumerate(input_data_folders):    
    print(ind_dataset)
    # Load data; extract UMI-counts and cell-IDs
    umi_counts_df = pd.read_csv(os.path.join(input_data_folder, 'Gene_table.txt'), header=0,
                                        index_col=0, sep='\t')
    cell_ids = list(umi_counts_df.columns)
    umi_counts = umi_counts_df.values
    del umi_counts_df
    
    cell_ids_lst.append(cell_ids)
    umi_counts_lst.append(umi_counts)

In [ ]:
pca_projected_lst = []
PCA_COMPS = [2]
if N_PCS_UMAP > 0:
    PCA_COMPS.append(N_PCS_UMAP)
if N_PCS_PHATE > 0:
    PCA_COMPS.append(N_PCS_PHATE)
if N_PCS_DTNE > 0:
    PCA_COMPS.append(N_PCS_DTNE)

PCA_COMPS = np.unique(PCA_COMPS)

logp1_lst = []

for ind_dataset, input_data_folder in enumerate(input_data_folders):
    print(ind_dataset)
    umi_counts = umi_counts_lst[ind_dataset]
    # Perform logp1
    logp1 = do_logp1(umi_counts)

    # Perform PCA to 2 components for visualization and to 10 components for subsequent UMAP
    pca_projected = do_pca(logp1, n_comps_list=PCA_COMPS)
    # del logp1
    logp1_lst.append(logp1)
    pca_projected_lst.append(pca_projected)

In [ ]:
if 'umap' in methods:
    umap_projected_lst = []
    for ind_dataset, input_data_folder in enumerate(input_data_folders):   
        print(ind_dataset)
        if N_PCS_UMAP < 0:
            preprocessed = logp1_lst[ind_dataset]
        else:
            preprocessed = pca_projected_lst[ind_dataset][N_PCS_UMAP]
        # Perform UMAP on the larger-number-of-components PCA
        umap_projected = {}
        umap_projected[N_PCS_UMAP] = fit_umap(preprocessed, random_state=None, n_neighbors=15, min_dist=0.1,
                                           n_components=2,
                                           metric='euclidean',
                                           make_plot=False, title='')
        umap_projected_lst.append(umap_projected)

In [ ]:
if 'phate' in methods:
    phate_projected_lst = []
    for ind_dataset, input_data_folder in enumerate(input_data_folders):   
        print(ind_dataset)
        # Perform PHATE on the logp1-transformed data
        if N_PCS_PHATE < 0:
            preprocessed = logp1_lst[ind_dataset]
        else:
            preprocessed = pca_projected_lst[ind_dataset][N_PCS_PHATE]
        phate_projected = {}
        phate_projected[N_PCS_PHATE] = fit_phate(preprocessed)
        phate_projected_lst.append(phate_projected)

In [ ]:
if 'DTNE' in methods:
    dtne_projected_lst = []
    for ind_dataset, input_data_folder in enumerate(input_data_folders):   
        print(ind_dataset)
        # Perform DTNE on the logp1-transformed data
        if N_PCS_DTNE < 0:
            preprocessed = logp1_lst[ind_dataset]
        else:
            preprocessed = pca_projected_lst[ind_dataset][N_PCS_DTNE]
        dtne_projected = {}
        dtne_projected[N_PCS_DTNE] = fit_DTNE(preprocessed)
        dtne_projected_lst.append(dtne_projected)

## Create pairwise distance plots

In [ ]:
# Read in ground truth squared pairwise distances (divided by the number of dimensions)
true_dists_lst = []
# selected_gene_inds_lst = []
for ind_dataset, input_data_folder in enumerate(input_data_folders):
    print(ind_dataset)
    delta_gc_true = pd.read_csv(os.path.join(input_data_folder, 'delta_true.txt'), header=None,
                                index_col=None, sep='\t').values
    
    num_dims = delta_gc_true.shape[0]
    true_dists = distance.pdist(delta_gc_true.T, metric='sqeuclidean')/num_dims
    true_dists_lst.append(true_dists)

In [ ]:
# INCLUDE_SANITY=True
if 'sanity' in methods:
    sanity_lst = []
    for ind_dataset, input_data_folder in enumerate(input_data_folders):
        print(ind_dataset)
        dataset_id = dataset_ids[ind_dataset]
        sanity_path = os.path.join(PATH_TO_PAPER_FIGURES, 'paper_figures_datasets', dataset_id, 'Sanity')
        sanity_dists = pd.read_csv(os.path.join(sanity_path, 'cell_cell_distance_with_errorbar_avzscore_gt_1.txt'), header=None,
                                   index_col=None, sep='\t').values.flatten()
        sanity_lst.append(sanity_dists)

In [ ]:
# Calculate distances on tree
if 'bonsai' in methods:
    bonsai_dists_lst = []
    for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
        print(ind_dataset)
        cell_ids = cell_ids_lst[ind_dataset]
        tree_results = os.path.join(bonsai_results, find_latest_tree_folder(bonsai_results))
        bonsai_dists = get_pdists_on_tree(os.path.join(tree_results, 'tree.nwk'), cell_ids)
        bonsai_dists_lst.append(bonsai_dists)

In [ ]:
# Calculate distances on tree
if 'bonsai-log1p' in methods:
    bonsai_log1p_dists_lst = []
    for ind_dataset, bonsai_results in enumerate(bonsai_logp1_results_folders):
        print(ind_dataset)
        cell_ids = cell_ids_lst[ind_dataset]
        tree_results = os.path.join(bonsai_results, find_latest_tree_folder(bonsai_results))
        bonsai_dists = get_pdists_on_tree(os.path.join(tree_results, 'tree.nwk'), cell_ids)
        bonsai_log1p_dists_lst.append(bonsai_dists)

In [ ]:
if 'pca' in methods:
    pca_dists_lst = []
    for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
        print(ind_dataset)
        pca_projected = pca_projected_lst[ind_dataset]
        for n_comps, pca_proj in pca_projected.items():
            if n_comps != 2:
                continue
            pca_dists = distance.pdist(pca_proj.T, metric='sqeuclidean') / 2
            pca_dists_lst.append(pca_dists)

In [ ]:
if 'umap' in methods:
    umap_dists_lst = []
    for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
        print(ind_dataset)
        umap_projected = umap_projected_lst[ind_dataset]
        umap_proj = umap_projected[N_PCS_UMAP]
        umap_dists = distance.pdist(umap_proj.T, metric='sqeuclidean') / 2
        umap_dists_lst.append(umap_dists)

In [ ]:
if 'phate' in methods:
    phate_dists_lst = []
    for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
        print(ind_dataset)
        phate_projected = phate_projected_lst[ind_dataset]
        phate_proj = phate_projected[N_PCS_PHATE]
        phate_dists = distance.pdist(phate_proj.T, metric='sqeuclidean') / 2
        phate_dists_lst.append(phate_dists)

In [ ]:
if 'DTNE' in methods:
    dtne_dists_lst = []
    for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
        print(ind_dataset)
        dtne_projected = dtne_projected_lst[ind_dataset]
        dtne_proj = dtne_projected[N_PCS_DTNE]
        dtne_dists = distance.pdist(dtne_proj.T, metric='sqeuclidean') / 2
        dtne_dists_lst.append(dtne_dists)

In [ ]:
# Initialize some list for storing information about the different tools
tool_objcts_lst = []

## Plot everything in big figure without the box-plots

In [ ]:
figs_lst = []
axs_lst = []

print(methods)

ncols = len(methods)
print(ncols)
for ind_dataset, dataset in enumerate(dataset_descr_lst):
    print(dataset)
    print(ind_dataset)
    fig, axs = plt.subplots(nrows=2, ncols=ncols, figsize=(len(methods)*3, 6))
    # Make the 2nd row share y-axis with the first subplot in the 2nd row
    for i in range(1, ncols):  # columns 1 to 3
        axs[1, i].sharey(axs[1, 0])
    figs_lst.append(fig)
    axs_lst.append(axs)
    for ax in axs.flatten():
        ax.set_box_aspect(1)

    plt.tight_layout()
    plt.subplots_adjust(left=0, right=1.0, bottom=0.12, top=0.88)
    fig.suptitle(dataset_descr_lst[ind_dataset], fontsize=20)

In [ ]:
# Create Bonsai visualization
# Visualize the tree in the equal-daylight layout, with the correct celltype-annotation
for ind_dataset, bonvis_fig in enumerate(bonvis_fig_lst):
    dataset_fig = figs_lst[ind_dataset]
    dataset_axs = axs_lst[ind_dataset]
    bonvis_fig.bonvis_settings.transf_info.ax_lims = np.array([-1.01, 1.01, -1.01, 1.01])
    # bonvis_fig.update_figure(geometry='flat')
    bonvis_fig.update_figure(geometry='hyperbolic')
    bonvis_fig.create_figure(figsize=(6, 6), fig=dataset_fig, ax=dataset_axs[0, 1])
    dataset_descr = dataset_descr_lst[ind_dataset]
    if dataset_descr not in dataset_descr_lst_gt:
        dataset_axs[0,0].axis('off')

In [ ]:
# Create Bonsai visualization for standard preprocessing
# Visualize the tree in the equal-daylight layout, with the correct celltype-annotation
for ind_dataset, bonvis_fig in enumerate(bonvis_logp1_fig_lst):
    dataset_fig = figs_lst[ind_dataset]
    dataset_axs = axs_lst[ind_dataset]
    bonvis_fig.bonvis_settings.transf_info.ax_lims = np.array([-1.01, 1.01, -1.01, 1.01])
    # bonvis_fig.update_figure(geometry='flat')
    bonvis_fig.update_figure(geometry='hyperbolic')
    bonvis_fig.create_figure(figsize=(6, 6), fig=dataset_fig, ax=dataset_axs[0, -1])
    dataset_descr = dataset_descr_lst[ind_dataset]
    if dataset_descr not in dataset_descr_lst_gt:
        dataset_axs[0,0].axis('off')

In [ ]:
# Create Bonsai visualization for ground truth
# Visualize the tree in the equal-daylight layout, with the correct celltype-annotation
for ind_dataset_gt, bonvis_fig in enumerate(bonvis_fig_lst_gt):
    ind_dataset = dataset_descr_lst.index(dataset_descr_lst_gt[ind_dataset_gt])
    dataset_fig = figs_lst[ind_dataset]
    dataset_axs = axs_lst[ind_dataset]
    bonvis_fig.bonvis_settings.transf_info.ax_lims = np.array([-1.01, 1.01, -1.01, 1.01])
    # bonvis_fig.update_figure(geometry='flat')
    bonvis_fig.create_figure(figsize=(6, 6), fig=dataset_fig, ax=dataset_axs[0,0])

In [ ]:
annotation_folders = [os.path.join(PATH_TO_PAPER_FIGURES, 'paper_figures_datasets', dataset_id, 'annotation') for dataset_id in dataset_ids]

tools_lst = ['pca', 'umap', 'phate', 'DTNE']
tools_lst = [tool for tool in tools_lst if tool in methods]

# Make figure for 2D-PCA and 2D-UMAP
for ind_dataset, input_data_folder in enumerate(input_data_folders):
    print(ind_dataset)
    bonvis_fig = bonvis_fig_lst[ind_dataset]
    cats_to_color = bonvis_fig.bonvis_settings.node_style['annot_info'].annot_to_color

    proj_dct = {}
    if 'pca' in methods:
        pca_projected = pca_projected_lst[ind_dataset]
        proj_dct['pca'] = pca_projected[2]
    if 'umap' in methods:
        umap_projected = umap_projected_lst[ind_dataset]
        proj_dct['umap'] = umap_projected[N_PCS_UMAP]
    if 'phate' in methods:
        phate_projected = phate_projected_lst[ind_dataset]
        proj_dct['phate'] = phate_projected[N_PCS_PHATE]
    if 'DTNE' in methods:
        dtne_projected = dtne_projected_lst[ind_dataset]
        proj_dct['DTNE'] = dtne_projected[N_PCS_DTNE]
    data_descr = dataset_descr_lst[ind_dataset]
    cell_ids = cell_ids_lst[ind_dataset]
    
    # Read in annotation to get color information for the UMAP
    scData = SCData(onlyObject=True, dataset=dataset_ids[ind_dataset])
    scData.metadata.nCells = len(cell_ids)
    scData.metadata.cellIds = cell_ids
    print(os.path.join(input_data_folder, 'annotation'))
    annotation_df, feature_matrices = scData.get_annotations(annotation_folders[ind_dataset])
    special_annotations = {'Pseudobulk': 'annot_pseudobulk'}
    annotation_label = special_annotations[data_descr] if data_descr in special_annotations else 'annot_Celltype3'
    annotation_to_be_used = annotation_df[annotation_label]
    cats = np.sort(np.unique(annotation_to_be_used))
    colors = [cats_to_color[cat] for cat in annotation_to_be_used]

    fig = figs_lst[ind_dataset]

    for ind, tool in enumerate(tools_lst):
        tool_index = ind + 2
    
        ax = axs_lst[ind_dataset][0, tool_index]
        ax.set_box_aspect(1)
        plt.subplots_adjust(left=0, right=1.0, bottom=0, top=1.0)
        ax.axes.get_xaxis().set_visible(False)
        ax.axes.get_yaxis().set_visible(False)

        proj = proj_dct[tool]

        ax.scatter(proj[0, :], proj[1, :], s=5, c=colors)



In [ ]:
tools_lst = ['bonsai', 'pca', 'umap', 'phate', 'DTNE', "bonsai-log1p"]
tools_lst = [tool for tool in tools_lst if tool in methods]

fig, axs = plt.subplots(ncols=len(tools_lst))
tools_lst = [tool for tool in tools_lst if tool in methods]
bonsai_results = bonsai_results_folders[0]
ind_dataset = 0
true_dists = true_dists_lst[ind_dataset]
dists_lst = {}
if 'bonsai' in tools_lst:
    dists_lst['bonsai'] = bonsai_dists_lst
# if 'bonsai_log1p' in tools_lst:
if 'bonsai-log1p' in tools_lst:
    # dists_lst['bonsai_log1p'] = bonsai_log1p_dists_lst
    dists_lst['bonsai-log1p'] = bonsai_log1p_dists_lst
if 'pca' in tools_lst:
    dists_lst['pca'] = pca_dists_lst
if 'umap' in tools_lst:
    dists_lst['umap'] = umap_dists_lst
if 'phate' in tools_lst:
    dists_lst['phate'] = phate_dists_lst
if 'DTNE' in tools_lst:
    dists_lst['DTNE'] = dtne_dists_lst

for ind_tool, tool in enumerate(tools_lst):
    ax = axs[ind_tool]
    dists = dists_lst[tool][0]
    ax.scatter(true_dists, dists, s=2)

In [ ]:
# Create histogram of correlations figures
# tools_lst = ['sanity', 'bonsai', 'pca', 'umap', 'phate', 'DTNE']
tools_lst = ['sanity', 'bonsai', 'pca', 'umap', 'phate', 'DTNE', "bonsai-log1p"]
tools_lst = [tool for tool in tools_lst if tool in methods]
print(tools_lst)
print(methods)

RECALCULATE = True
RECALCULATE = RECALCULATE or not len(tool_objcts_lst)
if RECALCULATE:
    tool_objcts_lst = []
    

for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
    print("\n\nTreating dataset {}\n".format(dataset_descr_lst[ind_dataset]))

    true_dists = true_dists_lst[ind_dataset]

    tools_dists_dict = {}
    if 'bonsai' in tools_lst:
        tools_dists_dict['bonsai'] = bonsai_dists_lst[ind_dataset]
    if 'bonsai-log1p' in tools_lst:
        tools_dists_dict['bonsai-log1p'] = bonsai_log1p_dists_lst[ind_dataset]
    if 'pca' in tools_lst:
        tools_dists_dict['pca'] = pca_dists_lst[ind_dataset]
    if 'umap' in tools_lst:
        tools_dists_dict['umap'] = umap_dists_lst[ind_dataset]
    if 'phate' in tools_lst:
        tools_dists_dict['phate'] = phate_dists_lst[ind_dataset]
    if 'DTNE' in tools_lst:
        tools_dists_dict['DTNE'] = dtne_dists_lst[ind_dataset]
    if 'sanity' in methods:
        tools_dists_dict['sanity'] =  sanity_lst[ind_dataset]
    if RECALCULATE:
        tool_objcts = {}
        true_objct = Dataset(distances=true_dists_lst[ind_dataset], data_type='delta_true', data_id='delta_true', color_types=tools_lst)
        true_objct.true_dataset_ranks = None
        for ind_tool, tool in enumerate(tools_lst):
            data_id = tool + dataset_descr_lst[ind_dataset]
            tool_objcts[tool] = Dataset(distances=tools_dists_dict[tool], data_type=tool, data_id=data_id, color_types=tools_lst)
    else:
        tool_objcts = tool_objcts_lst[ind_dataset]
    for ind_tool, tool in enumerate(tools_lst):
        tool_objct = tool_objcts[tool]

        fig = figs_lst[ind_dataset]
        ax = axs_lst[ind_dataset][1, ind_tool]
        ax.set_box_aspect(1)
        n_neighbours_list = compare_nearest_neighbours_to_truth([true_objct, tool_objct], make_fig=False, max_neighbours=600, ax=None,
                                                 only_powers_of_2=True,
                                                 title='')
        avg_rel_diffs, R_vals = compare_pdists_to_truth_per_cell([true_objct, tool_objct], make_fig=True, axs=ax, set_lims=False, return_Rvals=True, XLABEL=False, YLABEL=False, flip_axes=True, first_title=' ', loglog_corr=False)
    if RECALCULATE:
        tool_objcts_lst.append(tool_objcts)

In [ ]:
nrows = 2
ncols = int(np.ceil(len(dataset_ids)/nrows))
fig_nn, axs_nn = plt.subplots(figsize=(3*ncols,3*nrows), nrows=nrows, ncols=ncols)
marker = '*-' if len(n_neighbours_list) < 40 else '-'

if len(dataset_ids) == 1:
    axs_nn = np.array([axs_nn])
axs_nn = axs_nn.flatten()
for ax in axs_nn:
    ax.axis('off')

for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
    ax = axs_nn[ind_dataset]
    ax.axis('on')
    dataset_descr = dataset_descr_lst[ind_dataset]
    for ind_tool, tool in enumerate(tools_lst):
        # if ('sanity' not in methods) and (tool == 'sanity'):
        #     continue
        tool_objct = tool_objcts_lst[ind_dataset][tool]
        cf_nn = tool_objct.correct_fractions_of_neighbours
        
        if tool != 'bonsai':
            ax.plot(n_neighbours_list, tool_objct.correct_fractions_of_neighbours, marker,
                    c=tool_objct.data_type_color,
                    label=tool, zorder=0)
        else:
            ax.plot(n_neighbours_list, tool_objct.correct_fractions_of_neighbours, marker,
                    c=tool_objct.data_type_color, linewidth=3, label=tool, zorder=1)
    

    if ind_dataset == 0:
        ax.legend(loc='lower right')
        ax.set_ylabel('Fraction of correct\nnearest neighbours')
    ax.set_xlabel('Number of nearest neighbours')
    ax.set_xscale('log')
    ax.set_ylim(0, 1)
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.ticklabel_format(style='plain', axis='x')

    ax.set_title(dataset_descr)
plt.tight_layout()

In [ ]:
# Create boxplots in separate figure as well, now grouped by dataset
tools_lst = ['sanity', 'bonsai', 'pca', 'umap', 'phate', 'DTNE', "bonsai-log1p"]
tools_lst = [tool for tool in tools_lst if tool in methods]

boxprops = dict(linewidth=0.05)
flierprops = dict(markersize=2, markeredgewidth=0.5)
medianprops = dict(color='black', linewidth=1)
nrows = 2 if not SINGLE_DATASET else 1
ncols = int(np.ceil(len(dataset_ids)/nrows))
fig_bp, axs_bp = plt.subplots(figsize=(3*ncols,3*nrows), nrows=nrows, ncols=ncols)

if len(dataset_ids) == 1:
    axs_bp = np.array([axs_bp])
axs_bp = axs_bp.flatten()
for ax in axs_bp:
    ax.axis('off')

for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
    ax = axs_bp[ind_dataset]
    ax.axis('on')
    dataset_descr = dataset_descr_lst[ind_dataset]
    pearsonRs = []
    tool_names = []
    for ind_tool, tool in enumerate(tools_lst):
        # if (not INCLUDE_SANITY) and (tool == 'sanity'):
        #     continue
        tool_objct = tool_objcts_lst[ind_dataset][tool]
        tool_names.append(tool)
        pearsonRs.append(tool_objct.pearsonRs)
    ax = axs_bp[ind_dataset]
    bplot = ax.boxplot(pearsonRs, whis=(5, 95), labels=tool_names, patch_artist=True, 
                           flierprops=flierprops, medianprops=medianprops, boxprops=boxprops)
    # fill with colors
    for ind_patch, patch in enumerate(bplot['boxes']):
        patch.set_facecolor(color=tool_objcts_lst[ind_dataset][tools_lst[ind_patch]].data_type_color)
    ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_ylim(-0.05,1.05)
    ax.set_title(dataset_descr)
    if ind_dataset == 0:
        ax.set_ylabel('Pearson R')
plt.tight_layout()

In [ ]:
# Change some titles and labels
tools_lst = ['sanity', 'bonsai', 'pca', 'umap', 'phate', 'DTNE', 'bonsai-log1p']
tools_lst = [tool for tool in tools_lst if tool in methods]

print(methods)
print(tools_lst)
for ind_fig, fig_obj in enumerate(figs_lst):
    axs = axs_lst[ind_fig]
#     axs[2,0].axis('off')
    if dataset_descr_lst[ind_fig] in dataset_descr_lst_gt:
        axs[0,0].set_title('Ground truth', fontsize=16)
    if 'bonsai' in tools_lst:
        axs[0, tools_lst.index('bonsai')].set_title('Bonsai', fontsize=16)
        axs[1, tools_lst.index('bonsai')].set_title('Bonsai', fontsize=16)
    if 'bonsai_log1p' in tools_lst:
        axs[0, tools_lst.index('bonsai-log1p')].set_title('Bonsai logp1', fontsize=16)
        axs[1, tools_lst.index('bonsai-log1p')].set_title('Bonsai logp1', fontsize=16)
    if 'pca' in tools_lst:
        axs[0, tools_lst.index('pca')].set_title('PCA', fontsize=16)
        axs[1, tools_lst.index('pca')].set_title('PCA', fontsize=16)
    if 'umap' in tools_lst:
        axs[0, tools_lst.index('umap')].set_title('UMAP', fontsize=16)
        axs[1, tools_lst.index('umap')].set_title('UMAP', fontsize=16)
    if 'phate' in tools_lst:
        axs[0, tools_lst.index('phate')].set_title('PHATE', fontsize=16)
        axs[1, tools_lst.index('phate')].set_title('PHATE', fontsize=16)
    if 'DTNE' in tools_lst:
        axs[0, tools_lst.index('DTNE')].set_title('DTNE', fontsize=16)
        axs[1, tools_lst.index('DTNE')].set_title('DTNE', fontsize=16)
    if 'sanity' in tools_lst:
        axs[1, 0].set_title('Sanity', fontsize=16)

    for ind_ax, ax in enumerate(axs[1,:]):
        ax.set_xlabel("Pearson R", fontsize=12)
    
    # Turn everything off except for the y-label for the top left figure.
    ax = axs[0,0]
    axs[0,0].axis('on')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    
    ax.tick_params(bottom=False, labelbottom=False)
    ax.tick_params(top=False, right=False)
    ax.tick_params(left=False, labelleft=False)


    axs[1,0].set_ylabel('Number of cells', fontsize=12)

    fig_obj.subplots_adjust(bottom=0.05, top=0.88, left=0.1, right=.9)
    fig_obj.text(0.03, 0.9, 'A)', fontsize=14, fontweight='bold', va='top', ha='left')
    fig_obj.text(0.03, 0.45, 'B)', fontsize=14, fontweight='bold', va='top', ha='left')


### Show figures

In [ ]:
ind=0
figs_lst[ind]

In [ ]:
ind=1
figs_lst[ind]

In [ ]:
ind=2
figs_lst[ind]

In [ ]:
ind=3
figs_lst[ind]

In [ ]:
ind=4
figs_lst[ind]

In [ ]:
ind=5
figs_lst[ind]

In [ ]:
ind=6
figs_lst[ind]

## Make big panel-figure with all visualizations

In [ ]:
%%capture
ncols = 5
nrows = len(dataset_descr_lst)
fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 3, 21), layout="constrained")
if nrows == 1:
    axs = axs[None, :]


for ind_dataset, dataset_descr in enumerate(dataset_descr_lst):
    axs[ind_dataset, 0].set_ylabel(dataset_descr)

axs[0,0].set_title('Ground truth', fontsize=16)
axs[0,1].set_title('Bonsai', fontsize=16)
axs[0,2].set_title('Bonsai log1p', fontsize=16)  


In [ ]:
# Create Bonsai visualization
# Visualize the tree in the equal-daylight layout, with the correct celltype-annotation
for ind_dataset, bonvis_fig in enumerate(bonvis_fig_lst):
    dataset_fig = fig
    dataset_ax = axs[ind_dataset, 1]
    bonvis_fig.update_figure()
    bonvis_fig.create_figure(figsize=(6, 6), fig=dataset_fig, ax=dataset_ax)
    dataset_descr = dataset_descr_lst[ind_dataset]
    plt.subplots_adjust(left=-0.10, right=1.1, bottom=-0.1, top=1.1)


In [ ]:
# Create Bonsai visualization with standard preprocessing
# Visualize the tree in the equal-daylight layout, with the correct celltype-annotation
for ind_dataset, bonvis_fig in enumerate(bonvis_logp1_fig_lst):
    dataset_fig = fig
    dataset_ax = axs[ind_dataset, 2]
    bonvis_fig.update_figure()
    bonvis_fig.create_figure(figsize=(6, 6), fig=dataset_fig, ax=dataset_ax)
    dataset_descr = dataset_descr_lst[ind_dataset]
    plt.subplots_adjust(left=-0.10, right=1.1, bottom=-0.1, top=1.1)


In [ ]:
# Create Bonsai visualization for ground truth
# Visualize the tree in the equal-daylight layout, with the correct celltype-annotation
for ind_dataset_gt, bonvis_fig in enumerate(bonvis_fig_lst_gt):
    ind_dataset = dataset_descr_lst.index(dataset_descr_lst_gt[ind_dataset_gt])
    dataset_fig = fig
    ax = axs[ind_dataset, 0]
    bonvis_fig.update_figure()
    bonvis_fig.create_figure(figsize=(6, 6), fig=dataset_fig, ax=ax)
    plt.subplots_adjust(left=0, right=1.0, bottom=0, top=1.0)

In [ ]:
# Add BoxPlots
tools_lst = ['bonsai', "bonsai-log1p", 'pca']
tools_clean_names = {"bonsai":"bonsai", "bonsai-log1p":"bonsai\nlog1p", "pca":"pca"}
tools_lst = [tool for tool in tools_lst if tool in methods]

boxprops = dict(linewidth=0.05)
flierprops = dict(markersize=2, markeredgewidth=0.5)
medianprops = dict(color='black', linewidth=1)

for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
    ax = axs[ind_dataset, 3]
    pearsonRs = []
    tool_names = []
    for ind_tool, tool in enumerate(tools_lst):
        # if (not INCLUDE_SANITY) and (tool == 'sanity'):
        #     continue
        tool_objct = tool_objcts_lst[ind_dataset][tool]
        tool_names.append(tool)
        pearsonRs.append(tool_objct.pearsonRs)
    
    bplot = ax.boxplot(pearsonRs, whis=(5, 95), labels=[tools_clean_names[tool_name] for tool_name in tool_names], patch_artist=True, 
                           flierprops=flierprops, medianprops=medianprops, boxprops=boxprops)

    
    # fill with colors
    for ind_patch, patch in enumerate(bplot['boxes']):
        patch.set_facecolor(color=tool_objcts_lst[ind_dataset][tools_lst[ind_patch]].data_type_color)
        
    ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation=0, ha='center')
    ax.set_ylim(-0.05,1.05)
    ax.set_ylabel('Pearson R')


In [ ]:
# Add knn
tools_lst = ['bonsai', "bonsai-log1p", 'pca']
tools_lst = [tool for tool in tools_lst if tool in methods]

boxprops = dict(linewidth=0.05)
flierprops = dict(markersize=2, markeredgewidth=0.5)
medianprops = dict(color='black', linewidth=1)

for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
    ax = axs[ind_dataset, 4]
    pearsonRs = []
    tool_names = []
    for ind_tool, tool in enumerate(tools_lst):
        # if (not INCLUDE_SANITY) and (tool == 'sanity'):
        #     continue
        tool_objct = tool_objcts_lst[ind_dataset][tool]
        cf_nn = tool_objct.correct_fractions_of_neighbours
        
        if tool != 'bonsai':
            ax.plot(n_neighbours_list, tool_objct.correct_fractions_of_neighbours, marker,
                    c=tool_objct.data_type_color,
                    label=tool, zorder=0)
        else:
            ax.plot(n_neighbours_list, tool_objct.correct_fractions_of_neighbours, marker,
                    c=tool_objct.data_type_color, linewidth=3, label=tool, zorder=1)

    

    ax.set_xlabel('Number of nearest neighbours')
    ax.set_xscale('log')
    ax.set_ylabel('Fraction of correct\nnearest neighbours')
    ax.set_ylim(0, 1)
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.ticklabel_format(style='plain', axis='x')
            

In [ ]:
for ind_dataset, dataset_descr in enumerate(dataset_descr_lst):
    ax = axs[ind_dataset, 0]
    ax.axis('on')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    
    ax.tick_params(bottom=False, labelbottom=False)
    ax.tick_params(top=False, right=False)
    ax.tick_params(left=False, labelleft=False)
    ax.set_ylabel(dataset_descr, fontsize=12)


In [ ]:
fig